# Dataset Insights (Resolution 1024)
This notebook summarizes a generated dataset of elevation-map paths and computes additional insights:
- path crossing heatmap
- elevation distributions
- path length distribution
- normalized relative elevation profile
- normalized Hausdorff distances
- identical path count


## 1. Setup and Configuration
Import libraries, configure solver, and load calibration and dataset summary files.


In [ ]:
import json
import logging
import sys
import multiprocessing
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm
import cv2
from loguru import logger as loguru_logger

try:
    from passage.generate import _resolve_solver_config
except Exception:
    # If import fails (e.g., running in notebook without package installed),
    # temporarily prepend local `src` to sys.path and retry.
    ROOT = Path("..").resolve()
    sys.path.insert(0, str(ROOT / "src"))
    from passage.generate import _resolve_solver_config

# Configure info logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Configure loguru for info level
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="INFO")


# Paths
CONFIG_PATH = ROOT / "config" / "config.yaml"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"
DATASET_DIR = ROOT / "outputs" / "dataset"
SUMMARY_PATH = DATASET_DIR / "summary.json"
DEFAULT_COST_MODEL = "elevation"

# Solver config
with open(CONFIG_PATH) as f:
    _config = yaml.safe_load(f) or {}
_solver_cfg = _config.get("solver", {})
SOLVER_CONFIG = _resolve_solver_config(_solver_cfg)

# Calibration
with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)
global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])

print(f"Global elevation range: [{global_min:.1f}, {global_max:.1f}]")

Global elevation range: [-430.0, 8767.0]


In [2]:
# Parameters (papermill)
RESOLUTION = 256
MAX_SAMPLES = None
MAX_WORKERS = 8
SAVE_WORLD_COVER = True

## 2. Dataset Generation
For each dataset sample, we load the stored paths and per-point elevations from metadata (no recomputation):
- **Straight-line path** between markers (baseline, generated analytically)
- **Stored no-obstacle path** (`paths.without_obstacles`) and **stored obstacle-aware path** (`paths.with_obstacles`)

Metadata now includes per-point elevations and the list of tiles used to build the crop. Metrics are computed for each sample using multiprocessing to speed up processing across CPU cores.

### Metric computations and explanations

- **Path Occurrence Heatmap**: for every sample, we increment a per-pixel counter for each pixel that belongs to the **with_obstacles** stored path. The resulting image shows frequent routing corridors.

- **Covered Tiles Visualization**: we count how often each source tile appears in `tiles_used` and visualize coverage in a lat/lon scatter plot, colored by usage frequency.

- **Start Point Elevation Distribution**: histogram of the elevation at the first pixel of each stored path. This helps understand where paths begin (e.g., high vs low terrain).

- **All Path Points Elevation Distribution**: histogram aggregating elevation values for every path pixel across all samples.

- **Path Length Distribution**: histogram of path lengths measured as number of pixels in the stored path.

- **Normalized Relative Path Elevation Profile**: for each stored path we subtract the start elevation (relative elevation) and resample the sequence to 100 uniformly spaced points in the normalized path domain [0, 1]. We then compute the mean and standard deviation across samples to show a typical elevation trend along a path.

- **Normalized Hausdorff Distance**: we compute the Hausdorff distance $H(P,Q)$ between two paths and normalize by the straight-line (marker) distance $D$ between start and end: normalized = $H(P,Q) / D$. We compute two variants:
  - **Clean vs Straight**: $P$ = path_without_obstacles, $Q$ = straight-line markers path. Measures terrain-induced deviation from straight.
  - **Obstacles vs Clean**: $P$ = path_with_obstacles, $Q$ = path_without_obstacles. Measures deviation specifically introduced by obstacles.

- **Identical Path Count**: number and percentage of samples where the stored obstacle-aware path exactly equals the stored no-obstacle path (exact pixel list equality).

In [3]:
# Dataset paths and summary (derived from parameters)
DATASET_DIR = ROOT / "outputs" / "dataset" / str(RESOLUTION)
METADATA_DIR = DATASET_DIR / "metadata"
TENSORS_DIR = DATASET_DIR / "tensors"
SUMMARY_PATH = DATASET_DIR / "summary.json"

with open(SUMMARY_PATH) as f:
    summary = json.load(f)

print("Dataset Summary:")
print(json.dumps(summary, indent=2))


def describe_stats(values, name):
    values = [v for v in values if v is not None]
    if not values:
        print(f"{name}: no data")
        return None
    arr = np.array(values, dtype=np.float64)
    stats = {
        "count": int(arr.size),
        "min": float(np.min(arr)),
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "max": float(np.max(arr)),
        "std": float(np.std(arr)),
    }
    print(f"{name}: {stats}")
    return stats


def get_marker_positions(resolution: int):
    pos1 = int(0.2 * resolution)
    pos2 = int(0.8 * resolution)
    return (pos1, pos1), (pos2, pos2)


def make_straight_path(start, end, num_points=100):
    r_vals = np.linspace(start[0], end[0], num_points)
    c_vals = np.linspace(start[1], end[1], num_points)
    return list(zip(r_vals, c_vals))


def marker_distance(start, end) -> float:
    return float(np.linalg.norm(np.array(end) - np.array(start)))


def hausdorff_distance(path1: list, path2: list) -> float:
    """Hausdorff distance between two paths (max of directed distances)."""
    if len(path1) == 0 or len(path2) == 0:
        return 0.0
    arr1 = np.array(path1, dtype=np.float64)
    arr2 = np.array(path2, dtype=np.float64)
    # Directed distances: for each point in A, min distance to any point in B
    dists_1 = np.sqrt(((arr1[:, None, :] - arr2[None, :, :]) ** 2).sum(axis=2))
    dists_2 = np.sqrt(((arr2[:, None, :] - arr1[None, :, :]) ** 2).sum(axis=2))
    d1 = float(np.max(np.min(dists_1, axis=1)))
    d2 = float(np.max(np.min(dists_2, axis=1)))
    return max(d1, d2)


def resample_profile(values: np.ndarray, target_len: int = 100) -> np.ndarray:
    if len(values) == 0:
        return np.zeros(target_len, dtype=np.float64)
    x_old = np.linspace(0, 1, len(values))
    x_new = np.linspace(0, 1, target_len)
    return np.interp(x_new, x_old, values)


num_samples = int(summary.get("generated_samples", 0))
if MAX_SAMPLES is not None:
    num_samples = min(num_samples, MAX_SAMPLES)

start_marker, end_marker = get_marker_positions(RESOLUTION)
straight_path = make_straight_path(start_marker, end_marker, num_points=200)
marker_dist = marker_distance(start_marker, end_marker)

# Accumulators
path_crossings_obs = np.zeros((RESOLUTION, RESOLUTION), dtype=np.int32)
path_crossings_clean = np.zeros((RESOLUTION, RESOLUTION), dtype=np.int32)
start_elevations = []
all_elevations = []
path_lengths = []
normalized_profiles = []
hausdorff_clean_straight = []
hausdorff_obs_clean = []
identical_paths_count = 0

# New diagnostics from metadata
area_ratios = []
area_vals = []
costs_with = []
costs_without = []
cost_diff_percents = []

# Timing accumulators
graph_prebuild_seconds = []
obstacle_generation_seconds = []
cost_compute_seconds = []

tile_counts = {}

# Parallel processing using multiprocessing
cpu_total = os.cpu_count() or 1
if MAX_WORKERS is None:
    MAX_WORKERS = max(1, cpu_total)
else:
    MAX_WORKERS = max(1, min(int(MAX_WORKERS), cpu_total))
print(f"MAX_WORKERS={MAX_WORKERS}")


def process_sample(
    idx,
    metadata_dir,
    tensors_dir,
    resolution,
    global_min,
    global_max,
    start_marker,
    end_marker,
    straight_path,
    marker_dist,
):
    metadata_path = metadata_dir / f"{idx:010d}.yaml"
    tensor_path = tensors_dir / f"{idx:010d}.npy"

    if not metadata_path.exists() or not tensor_path.exists():
        return {"processed": False}

    with open(metadata_path) as f:
        meta = yaml.safe_load(f)

    paths = meta.get("paths", {})
    path_clean_pixels = paths.get("without_obstacles", {}).get("pixels", [])
    path_obs_pixels = paths.get("with_obstacles", {}).get("pixels", [])

    path_clean = [(int(p["i"]), int(p["j"])) for p in path_clean_pixels]
    path_obs = [(int(p["i"]), int(p["j"])) for p in path_obs_pixels]

    if not path_clean or not path_obs:
        return {"processed": False}

    p_clean_arr = np.array(path_clean)
    p_obs_arr = np.array(path_obs)

    # mask points inside image bounds for both paths
    obs_mask = (
        (p_obs_arr[:, 0] >= 0)
        & (p_obs_arr[:, 0] < resolution)
        & (p_obs_arr[:, 1] >= 0)
        & (p_obs_arr[:, 1] < resolution)
    )
    clean_mask = (
        (p_clean_arr[:, 0] >= 0)
        & (p_clean_arr[:, 0] < resolution)
        & (p_clean_arr[:, 1] >= 0)
        & (p_clean_arr[:, 1] < resolution)
    )

    if not (obs_mask.any() and clean_mask.any()):
        return {"processed": False}

    valid_obs_pts = p_obs_arr[obs_mask]
    valid_clean_pts = p_clean_arr[clean_mask]

    # Elevations: prefer per-point metadata, fallback to tensor if missing
    obs_elevs = [p.get("elevation") for p in path_obs_pixels]
    if len(obs_elevs) == len(path_obs) and all(e is not None for e in obs_elevs):
        path_elevs = np.array(obs_elevs, dtype=np.float64)[obs_mask]
    else:
        tensor = np.load(str(tensor_path))
        elev_norm = tensor[:, :, 0]
        elevation_map = elev_norm * (global_max - global_min) + global_min
        path_elevs = elevation_map[valid_obs_pts[:, 0], valid_obs_pts[:, 1]]

    if len(path_elevs) == 0:
        return {"processed": False}

    start_elev = float(path_elevs[0])
    all_elevs = [float(x) for x in path_elevs]
    rel_elevs = path_elevs - start_elev
    norm_profile = resample_profile(rel_elevs, target_len=100).tolist()

    h_clean_straight = hausdorff_distance(path_clean, straight_path) / max(marker_dist, 1e-6)
    h_obs_clean = hausdorff_distance(path_obs, path_clean) / max(marker_dist, 1e-6)
    identical = path_obs == path_clean

    tile_info = meta.get("tile", {})
    tiles_used = (
        tile_info.get("overlapping")
        or tile_info.get("used")
        or ([tile_info.get("source")] if tile_info.get("source") else [])
    )

    # Compute area between paths (now performed in notebook rather than in generator)
    area_pixels = 0
    try:
        if len(path_clean) >= 3 and len(path_obs) >= 3:
            poly_pts = np.array(
                [(p[1], p[0]) for p in path_clean] + [(p[1], p[0]) for p in reversed(path_obs)],
                dtype=np.int32,
            )
            mask_poly = np.zeros((resolution, resolution), dtype=np.uint8)
            try:
                cv2.fillPoly(mask_poly, [poly_pts], 1)
                area_pixels = int(mask_poly.sum())
            except Exception:
                area_pixels = 0
    except Exception:
        area_pixels = 0

    area_ratio = float(area_pixels / (marker_dist * marker_dist)) if marker_dist > 0 else 0.0

    # Extract cost diagnostics (already computed in generator)
    cost_with = meta.get("paths", {}).get("with_obstacles", {}).get("cost")
    cost_without = meta.get("paths", {}).get("without_obstacles", {}).get("cost")
    cost_diff_pct = meta.get("paths", {}).get("with_obstacles", {}).get("cost_diff_percent")

    # Extract timing diagnostics
    timing = meta.get("timing", {}) or {}
    graph_pre = float(timing.get("graph_prebuild_seconds", 0.0))
    obs_gen = float(timing.get("obstacle_generation_seconds", 0.0))
    cost_comp = float(timing.get("cost_compute_seconds", 0.0))

    return {
        "processed": True,
        "valid_obs_pts": valid_obs_pts.tolist(),
        "valid_clean_pts": valid_clean_pts.tolist(),
        "start_elev": start_elev,
        "all_elevs": all_elevs,
        "path_length": len(path_obs),
        "norm_profile": norm_profile,
        "h_clean_straight": h_clean_straight,
        "h_obs_clean": h_obs_clean,
        "identical": identical,
        "tiles_used": tiles_used,
        "area_ratio": float(area_ratio),
        "area_pixels": int(area_pixels),
        "cost_with": cost_with,
        "cost_without": cost_without,
        "cost_diff_pct": cost_diff_pct,
        "graph_prebuild_seconds": graph_pre,
        "obstacle_generation_seconds": obs_gen,
        "cost_compute_seconds": cost_comp,
    }


def _process_sample_star(args):
    return process_sample(*args)

Dataset Summary:
{
  "resolution": 256,
  "requested_samples": 1000,
  "start_index": 0,
  "generated_samples": 1000,
  "successful_samples": 1000,
  "mean_time_per_sample_seconds": 5.9420027630000005,
  "tiles_count": 23993,
  "tiles_dirpath": "outputs/download/tiles",
  "global_min": -430.0,
  "global_max": 8767.0,
  "timestamp": "2026-01-19T16:32:44.306449"
}
MAX_WORKERS=8


In [4]:
# Run processing (parallel or sequential depending on MAX_WORKERS)
# This cell defines `processed` and builds all aggregate statistics.
args = [
    (
        idx,
        METADATA_DIR,
        TENSORS_DIR,
        RESOLUTION,
        global_min,
        global_max,
        start_marker,
        end_marker,
        straight_path,
        marker_dist,
    )
    for idx in range(num_samples)
]

results = []
if MAX_WORKERS == 1:
    print("Running sequential processing (MAX_WORKERS==1)")
    for a in tqdm(args, total=len(args)):
        results.append(process_sample(*a))
else:
    pool_size = MAX_WORKERS
    print(f"Running with {pool_size} workers")
    with multiprocessing.Pool(processes=pool_size) as pool:
        for r in tqdm(pool.imap_unordered(_process_sample_star, args), total=len(args)):
            results.append(r)

# Filter successful processed entries
processed = [r for r in results if r.get("processed")]
samples_processed = len(processed)
print(f"Processed {samples_processed}/{len(results)} samples")

# Reset accumulators to avoid duplicate counts on re-run
start_elevations = []
all_elevations = []
path_lengths = []
normalized_profiles = []
hausdorff_clean_straight = []
hausdorff_obs_clean = []
identical_paths_count = 0

area_ratios = []
area_vals = []
costs_with = []
costs_without = []
cost_diff_percents = []

graph_prebuild_seconds = []
obstacle_generation_seconds = []
cost_compute_seconds = []

tile_counts = {}
path_crossings_obs = np.zeros((RESOLUTION, RESOLUTION), dtype=np.int32)
path_crossings_clean = np.zeros((RESOLUTION, RESOLUTION), dtype=np.int32)

# Aggregate tile usage, build heatmaps and other statistics
for entry in processed:
    # tiles
    for tile in entry.get("tiles_used", []) or []:
        if not tile:
            continue
        name = tile.get("name") or f"{tile.get('lat')}_{tile.get('lon')}"
        if name not in tile_counts:
            tile_counts[name] = {"lat": tile.get("lat"), "lon": tile.get("lon"), "count": 0}
        tile_counts[name]["count"] += 1

    # heatmaps: increment counts for obstacle path and clean path
    for r, c in entry.get("valid_obs_pts", []) or []:
        path_crossings_obs[int(r), int(c)] += 1
    for r, c in entry.get("valid_clean_pts", []) or []:
        path_crossings_clean[int(r), int(c)] += 1

    # basic stats
    start_elevations.append(entry.get("start_elev"))
    all_elevations.extend(entry.get("all_elevs", []) or [])
    path_lengths.append(entry.get("path_length") or 0)
    normalized_profiles.append(entry.get("norm_profile") or [])
    hausdorff_clean_straight.append(entry.get("h_clean_straight") or 0.0)
    hausdorff_obs_clean.append(entry.get("h_obs_clean") or 0.0)
    if entry.get("identical"):
        identical_paths_count += 1

    # new diagnostics
    ar = entry.get("area_ratio")
    if ar is not None:
        area_ratios.append(float(ar))
    ap = entry.get("area_pixels")
    if ap is not None:
        area_vals.append(int(ap))

    cw = entry.get("cost_with")
    if cw is not None:
        costs_with.append(float(cw))
    cwo = entry.get("cost_without")
    if cwo is not None:
        costs_without.append(float(cwo))
    cd = entry.get("cost_diff_pct")
    if cd is not None:
        cost_diff_percents.append(float(cd))

    # timing diagnostics
    gp = entry.get("graph_prebuild_seconds")
    if gp is not None:
        graph_prebuild_seconds.append(float(gp))
    og = entry.get("obstacle_generation_seconds")
    if og is not None:
        obstacle_generation_seconds.append(float(og))
    cc = entry.get("cost_compute_seconds")
    if cc is not None:
        cost_compute_seconds.append(float(cc))

# Covered Tiles Visualization: build a 180x360 (1° per pixel) grid image from metadata overlapping tiles
# Zero-based full-globe grid: rows=180 (lat), cols=360 (lon); lat range [-90,89], lon range [-180,179]
if tile_counts:
    grid = np.zeros((180, 360), dtype=np.int32)
    for name, t in tile_counts.items():
        lat = t.get("lat")
        lon = t.get("lon")
        cnt = int(t.get("count", 0))
        if lat is None or lon is None:
            continue
        lat_i = int(round(lat))
        lon_j = int(round(lon))
        # clamp to valid ranges
        lat_i = max(-90, min(89, lat_i))
        lon_j = max(-180, min(179, lon_j))
        i = 89 - lat_i  # row 0 = northmost lat (89)
        j = lon_j + 180
        grid[i, j] = cnt

    if grid.sum() == 0:
        print("No tile coverage to build 180x360 grid.")
    else:
        # Log-normalize non-zero counts for display
        logg = np.log1p(grid.astype(np.float64))
        nz = logg[grid > 0]
        if nz.size == 0:
            print("No nonzero counts after building grid.")
        else:
            vmin = float(nz.min())
            vmax = float(nz.max())

            # Create masked array for display (zeros shown as white)
            masked_logg = np.ma.masked_where(grid == 0, logg)

            # Display with matplotlib
            fig, ax = plt.subplots(figsize=(14, 7))
            cmap = cm.viridis.copy()
            cmap.set_bad(color="white")  # zeros shown as white

            # Use extent to set lat/lon axes
            extent = [-180, 179, -90, 89]  # [left, right, bottom, top]
            im = ax.imshow(
                masked_logg,
                origin="upper",
                extent=extent,
                cmap=cmap,
                aspect="equal",
                vmin=vmin if vmax > vmin else 0,
                vmax=vmax if vmax > vmin else 1,
            )

            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.set_title("Covered Tiles (180x360 grid, 1° per pixel) — zeros shown as white")

            # Add colorbar
            cbar = plt.colorbar(im, ax=ax, shrink=0.6, label="log1p(count)")

            plt.tight_layout()
            plt.show()

            # Save the image and raw counts
            if SAVE_WORLD_COVER:
                out_dir = ROOT / "outputs" / "dataset" / str(RESOLUTION)
                out_dir.mkdir(parents=True, exist_ok=True)
                img_path = out_dir / "world_cover_map.png"
                npy_path = out_dir / "world_cover_tiles.npy"

                # Save figure
                fig_save, ax_save = plt.subplots(figsize=(14, 7))
                im_save = ax_save.imshow(
                    masked_logg,
                    origin="upper",
                    extent=extent,
                    cmap=cmap,
                    aspect="equal",
                    vmin=vmin if vmax > vmin else 0,
                    vmax=vmax if vmax > vmin else 1,
                )
                ax_save.set_xlabel("Longitude")
                ax_save.set_ylabel("Latitude")
                ax_save.set_title("Covered Tiles (180x360 grid, 1° per pixel) — zeros shown as white")
                plt.colorbar(im_save, ax=ax_save, shrink=0.6, label="log1p(count)")
                plt.tight_layout()
                fig_save.savefig(str(img_path), dpi=150)
                plt.close(fig_save)

                np.save(str(npy_path), grid)
                print(f"Saved world cover image to {img_path} and raw counts to {npy_path}")
else:
    print("No tile coverage data available.")

Running with 8 workers


100%|██████████| 1000/1000 [00:05<00:00, 172.59it/s]


Processed 1000/1000 samples


Saved world cover image to redacted-other-home-02/projects/PASSAGE/passage/outputs/dataset/256/world_cover_map.png and raw counts to redacted-other-home-02/projects/PASSAGE/passage/outputs/dataset/256/world_cover_tiles.npy


In [5]:
# Diagnostics are now appended inside the main aggregation cell above.
# This cell intentionally left as a no-op placeholder.

## 3. Dataset Summary
Report basic statistics: requested vs. processed samples, and success rate.


In [6]:
summary_stats = {
    "resolution": RESOLUTION,
    "requested_samples": summary.get("requested_samples"),
    "generated_samples": summary.get("generated_samples"),
    "successful_samples": summary.get("successful_samples"),
    "processed_samples": len(processed),
    "processed_rate": (len(processed) / max(num_samples, 1)) * 100,
}

summary_df = pd.DataFrame([summary_stats])
summary_df

,resolution,requested_samples,generated_samples,successful_samples,processed_samples,processed_rate
0,256,1000,1000,1000,1000,100.0


## 4. Covered Tiles Visualization
Each point represents a source tile used to build the elevation crops. The color indicates how many samples used the tile.

In [7]:
# Covered Tiles Visualization: build a 180x360 (1° per pixel) grid image from metadata overlapping tiles
# Zero-based full-globe grid: rows=180 (lat), cols=360 (lon); lat range [-90,89], lon range [-180,179]
if tile_counts:
    grid = np.zeros((180, 360), dtype=np.int32)
    for name, t in tile_counts.items():
        lat = t.get("lat")
        lon = t.get("lon")
        cnt = int(t.get("count", 0))
        if lat is None or lon is None:
            continue
        lat_i = int(round(lat))
        lon_j = int(round(lon))
        # clamp to valid ranges
        lat_i = max(-90, min(89, lat_i))
        lon_j = max(-180, min(179, lon_j))
        i = 89 - lat_i  # row 0 = northmost lat (89)
        j = lon_j + 180
        grid[i, j] = cnt

    if grid.sum() == 0:
        print("No tile coverage to build 180x360 grid.")
    else:
        # Log-normalize non-zero counts for display
        logg = np.log1p(grid.astype(np.float64))
        nz = logg[grid > 0]
        if nz.size == 0:
            print("No nonzero counts after building grid.")
        else:
            vmin = float(nz.min())
            vmax = float(nz.max())

            # Create masked array for display (zeros shown as white)
            masked_logg = np.ma.masked_where(grid == 0, logg)

            # Display with matplotlib
            fig, ax = plt.subplots(figsize=(14, 7))
            cmap = cm.viridis.copy()
            cmap.set_bad(color="white")  # zeros shown as white

            # Use extent to set lat/lon axes
            extent = [-180, 179, -90, 89]  # [left, right, bottom, top]
            im = ax.imshow(
                masked_logg,
                origin="upper",
                extent=extent,
                cmap=cmap,
                aspect="equal",
                vmin=vmin if vmax > vmin else 0,
                vmax=vmax if vmax > vmin else 1,
            )

            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.set_title("Covered Tiles (180x360 grid, 1° per pixel) — zeros shown as white")

            # Add colorbar
            cbar = plt.colorbar(im, ax=ax, shrink=0.6, label="log1p(count)")

            plt.tight_layout()
            plt.show()

            # Save the image and raw counts
            if SAVE_WORLD_COVER:
                out_dir = ROOT / "outputs" / "dataset" / str(RESOLUTION)
                out_dir.mkdir(parents=True, exist_ok=True)
                img_path = out_dir / "world_cover_map.png"
                npy_path = out_dir / "world_cover_tiles.npy"

                # Save figure
                fig_save, ax_save = plt.subplots(figsize=(14, 7))
                im_save = ax_save.imshow(
                    masked_logg,
                    origin="upper",
                    extent=extent,
                    cmap=cmap,
                    aspect="equal",
                    vmin=vmin if vmax > vmin else 0,
                    vmax=vmax if vmax > vmin else 1,
                )
                ax_save.set_xlabel("Longitude")
                ax_save.set_ylabel("Latitude")
                ax_save.set_title("Covered Tiles (180x360 grid, 1° per pixel) — zeros shown as white")
                plt.colorbar(im_save, ax=ax_save, shrink=0.6, label="log1p(count)")
                plt.tight_layout()
                fig_save.savefig(str(img_path), dpi=150)
                plt.close(fig_save)

                np.save(str(npy_path), grid)
                print(f"Saved world cover image to {img_path} and raw counts to {npy_path}")
else:
    print("No tile coverage data available.")

Saved world cover image to redacted-other-home-02/projects/PASSAGE/passage/outputs/dataset/256/world_cover_map.png and raw counts to redacted-other-home-02/projects/PASSAGE/passage/outputs/dataset/256/world_cover_tiles.npy


## 4. Path Occurrence Heatmaps
Two heatmaps aggregate pixel crossing counts across the dataset:
- Left: Clean (no obstacles) stored path crossings
- Right: With Obstacles stored path crossings


In [8]:
# Path occurrence heatmaps (log-scaled) using matplotlib
clean_log = np.where(path_crossings_clean > 0, np.log1p(path_crossings_clean), np.nan)
obs_log = np.where(path_crossings_obs > 0, np.log1p(path_crossings_obs), np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(clean_log, cmap="hot", origin="upper")
axes[0].set_title("Path Crossing Heatmap - Clean (log scale)")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], label="log(1+count)")

im2 = axes[1].imshow(obs_log, cmap="hot", origin="upper")
axes[1].set_title("Path Crossing Heatmap - With Obstacles (log scale)")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], label="log(1+count)")

fig.suptitle(f"Path Crossing Heatmaps ({samples_processed} samples)")
plt.tight_layout()
plt.show()

## 5. Start Point Elevation Distribution
Histogram of the absolute elevation at the first point of each obstacle-aware path.

In [9]:
vals = [v for v in start_elevations if v is not None]
if vals:
    describe_stats(vals, "Start Point Elevations")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=30, edgecolor="black")
    plt.xlabel("Elevation (m)")
    plt.ylabel("Count")
    plt.title("Start Point Elevations")
    plt.tight_layout()
    plt.show()
else:
    print("No start elevations available.")

Start Point Elevations: {'count': 1000, 'min': -429.0, 'mean': 4227.161, 'median': 4347.5, 'max': 8764.0, 'std': 2605.489780651423}


## 6. All Path Points Elevation Distribution
Histogram of all elevations visited by obstacle-aware paths.

In [10]:
vals = [v for v in all_elevations if v is not None]
if vals:
    describe_stats(vals, "All Path Point Elevations")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=50, edgecolor="black")
    plt.xlabel("Elevation (m)")
    plt.ylabel("Count")
    plt.title("All Path Point Elevations")
    plt.tight_layout()
    plt.show()
else:
    print("No path elevation data available.")

All Path Point Elevations: {'count': 148568, 'min': -429.0, 'mean': 3752.9821630499164, 'median': 4043.0, 'max': 8764.0, 'std': 2317.25446276096}


## 7. Path Length Distribution
Histogram of obstacle-aware path lengths (number of pixels).

In [11]:
vals = [v for v in path_lengths if v is not None]
if vals:
    describe_stats(vals, "Path Lengths (pixels)")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=30, edgecolor="black")
    plt.xlabel("Length (pixels)")
    plt.ylabel("Count")
    plt.title("Path Lengths")
    plt.tight_layout()
    plt.show()
else:
    print("No path length data available.")

Path Lengths (pixels): {'count': 1000, 'min': 3.0, 'mean': 148.568, 'median': 143.5, 'max': 444.0, 'std': 74.1924212841177}


## 8. Normalized Relative Path Elevation Profile
Each path elevation sequence is shifted by its start elevation and resampled to a common length $[0,1]$.
We then plot the mean and standard deviation across samples.

In [12]:
if normalized_profiles:
    profiles_arr = np.array(normalized_profiles, dtype=np.float64)
    mean_profile = np.mean(profiles_arr, axis=0)
    std_profile = np.std(profiles_arr, axis=0)
    x_axis = np.linspace(0, 1, profiles_arr.shape[1])

    plt.figure(figsize=(10, 6))
    plt.plot(x_axis, mean_profile, "b-", linewidth=2, label="Mean Relative Elevation")
    plt.fill_between(
        x_axis, mean_profile - std_profile, mean_profile + std_profile, alpha=0.2, color="blue", label="Std Dev"
    )
    plt.xlabel("Normalized Path Progress")
    plt.ylabel("Relative Elevation (m)")
    plt.title("Normalized Relative Elevation Profile")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No normalized profiles available.")

## 9. Hausdorff Distance: Path vs. Straight Line
Normalized Hausdorff distance between the optimal (no-obstacle) path and the straight-line path between markers. This measures how much terrain elevation causes deviations from a straight line.

In [13]:
clean_vals = [v for v in hausdorff_clean_straight if v is not None and v > 0]
if clean_vals:
    describe_stats(clean_vals, "Normalized Hausdorff: Clean vs Straight")
    plt.figure(figsize=(8, 5))
    plt.hist(clean_vals, bins=30, edgecolor="black")
    plt.xlabel("Normalized Distance")
    plt.ylabel("Count")
    plt.title("Normalized Hausdorff: Clean vs Straight Line (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive Hausdorff data for clean vs straight.")

Normalized Hausdorff: Clean vs Straight: {'count': 1000, 'min': 0.15089407033483918, 'mean': 0.6119998113800598, 'median': 0.621559369757894, 'max': 1.137330025315357, 'std': 0.18261687008574884}


## 10. Hausdorff Distance: Obstacles vs. No Obstacles
Normalized Hausdorff distance between the obstacle-aware path and the no-obstacle path on the same elevation map.

In [14]:
obs_vals = [v for v in hausdorff_obs_clean if v is not None and v > 0]
if obs_vals:
    describe_stats(obs_vals, "Normalized Hausdorff: Obstacles vs Clean")
    plt.figure(figsize=(8, 5))
    plt.hist(obs_vals, bins=100, edgecolor="black")
    plt.xlabel("Normalized Distance")
    plt.ylabel("Count")
    plt.title("Normalized Hausdorff: Obstacles vs No Obstacles (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive Hausdorff data for obstacles vs clean.")

Normalized Hausdorff: Obstacles vs Clean: {'count': 460, 'min': 0.004621612948931683, 'mean': 0.11734388065660807, 'median': 0.09776612382410554, 'max': 0.5302180578394005, 'std': 0.08906635384439894}


## 11. Area Ratio & Cost Difference
Area ratio = (area between paths in pixels) / (marker distance in pixels)^2. Cost difference = 100 * (cost_with - cost_without) / cost_without.

In [15]:
# Area ratio distribution and cost difference visualization
area_pos = [v for v in area_ratios if v is not None and v > 0]
if area_pos:
    describe_stats(area_pos, "Area Ratio (positive)")
    plt.figure(figsize=(8, 5))
    plt.hist(area_pos, bins=40, edgecolor="black")
    plt.xlabel("Area Ratio")
    plt.ylabel("Count")
    plt.title("Area Ratio: (area between paths) / (marker_dist^2) (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive area ratio data available.")

# Cost difference percentage distribution
cd_vals = [v for v in cost_diff_percents if v is not None and v > 0]
if cd_vals:
    describe_stats(cd_vals, "Cost Difference Percent (positive)")
    plt.figure(figsize=(8, 5))
    plt.hist(cd_vals, bins=50, edgecolor="black")
    plt.xlabel("Percent")
    plt.ylabel("Count")
    plt.title("Cost Difference Percent: 100*(cost_with - cost_without)/cost_without")
    plt.tight_layout()
    plt.show()
else:
    print("No cost difference percentage data available.")

Area Ratio (positive): {'count': 1000, 'min': 6.407791874919901e-05, 'mean': 0.02278245546584647, 'median': 0.0046349694561920625, 'max': 0.5569652697680378, 'std': 0.04350501641805845}


Cost Difference Percent (positive): {'count': 446, 'min': 1.3077408550423968e-14, 'mean': 55.696954011986314, 'median': 13.56874683314333, 'max': 5251.935334641752, 'std': 267.3350099194747}


## 11b. Timing Summary

Plot per-sample timing distributions to identify hotspots:
- `graph_prebuild_seconds`
- `obstacle_generation_seconds`
- `cost_compute_seconds`


In [16]:
# Timing distributions and stats
gp_vals = [v for v in graph_prebuild_seconds if v is not None and v >= 0]
og_vals = [v for v in obstacle_generation_seconds if v is not None and v >= 0]
cc_vals = [v for v in cost_compute_seconds if v is not None and v >= 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if gp_vals:
    axes[0].hist(gp_vals, bins=30, edgecolor="black")
    axes[0].set_title("Graph prebuild (s)")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Count")
else:
    axes[0].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[0].set_title("Graph prebuild (s)")

if og_vals:
    axes[1].hist(og_vals, bins=30, edgecolor="black")
    axes[1].set_title("Obstacle generation (s)")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Count")
else:
    axes[1].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[1].set_title("Obstacle generation (s)")

if cc_vals:
    axes[2].hist(cc_vals, bins=30, edgecolor="black")
    axes[2].set_title("Cost compute (s)")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_ylabel("Count")
else:
    axes[2].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[2].set_title("Cost compute (s)")

fig.suptitle("Per-sample Timing Distributions")
plt.tight_layout()
plt.show()

if gp_vals:
    describe_stats(gp_vals, "Graph prebuild seconds")
if og_vals:
    describe_stats(og_vals, "Obstacle generation seconds")
if cc_vals:
    describe_stats(cc_vals, "Cost compute seconds")

if not (gp_vals or og_vals or cc_vals):
    print("No timing data available.")

Graph prebuild seconds: {'count': 1000, 'min': 0.0, 'mean': 0.0, 'median': 0.0, 'max': 0.0, 'std': 0.0}
Obstacle generation seconds: {'count': 1000, 'min': 0.005633831024169922, 'mean': 0.13168282985687255, 'median': 0.11726415157318115, 'max': 0.5120246410369873, 'std': 0.09043686044858612}
Cost compute seconds: {'count': 1000, 'min': 0.00039124488830566406, 'mean': 0.0031532323360443115, 'median': 0.0025037527084350586, 'max': 0.0458528995513916, 'std': 0.0025668379863288547}


## 12. Identical Path Count
Number of cases where the obstacle-aware path is exactly the same as the no-obstacle path.

In [17]:
if samples_processed > 0:
    pct_identical = (identical_paths_count / samples_processed) * 100
else:
    pct_identical = 0.0

print(f"Identical paths: {identical_paths_count} / {samples_processed} ({pct_identical:.1f}%)")

Identical paths: 540 / 1000 (54.0%)
